# Autonomous Experiment Loop

This notebook launches the autonomous Claude-driven experiment loop on your RunPod pod.

**How it works:**
1. Claude analyzes previous results and proposes hyperparameter changes
2. A GPT model trains for ~5 minutes with the new hyperparameters
3. The validation BPB (bits per byte) is evaluated
4. If the result improves on the best so far, it is kept; otherwise it is discarded
5. Repeat for up to `MAX_EXPERIMENTS` iterations

**Prerequisites:** Run `01_quick_start.ipynb` first to set up the environment, install dependencies, and prepare the data.

In [ ]:
# Configuration -- edit these values as needed
MAX_EXPERIMENTS = 20  # Adjust as needed
TAG = "runpod"
DATASET = "climbmix"
RESULTS_PATH = f"/workspace/autoresearch-unified/results/{DATASET}/results.tsv"

import os
os.chdir("/workspace/autoresearch-unified")
os.environ.setdefault("AUTORESEARCH_BACKEND", "cuda")
os.environ.setdefault("AUTORESEARCH_CACHE_DIR", "/workspace/.cache/autoresearch")
os.environ.setdefault("PYTHONPATH", "/workspace/autoresearch-unified")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first (run notebook 01)"
print(f"Config: max={MAX_EXPERIMENTS}, tag={TAG}, results={RESULTS_PATH}")

In [ ]:
# Option A: Run in foreground (blocks this cell, shows live output)
import sys
sys.path.insert(0, "/workspace/autoresearch-unified")
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
from tui.headless import run_headless
success = run_headless(tag=TAG, max_experiments=MAX_EXPERIMENTS, results_path=RESULTS_PATH, dataset_name=DATASET)
print(f"\nRun {'succeeded' if success else 'failed'}.")

In [ ]:
# Option B: Run in background (survives kernel restart)
import subprocess, sys
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)
cmd = f"nohup {sys.executable} -m tui.headless --tag {TAG} --max {MAX_EXPERIMENTS} --results {RESULTS_PATH} --dataset {DATASET} > /workspace/agent.log 2>&1 &"
subprocess.run(cmd, shell=True, cwd="/workspace/autoresearch-unified")
print("Agent launched in background. Check /workspace/agent.log for output.")
print("Use the monitoring cell below to track progress.")

In [ ]:
# Live monitoring -- polls results.tsv every 30 seconds
import time, os, pandas as pd
from IPython.display import display, clear_output

while True:
    clear_output(wait=True)
    if os.path.exists(RESULTS_PATH):
        df = pd.read_csv(RESULTS_PATH, sep="\t")
        kept = df[df["status"] == "keep"]
        discarded = df[df["status"] == "discard"]
        crashed = df[df["status"] == "crash"]
        baseline = df[df["status"] == "baseline"]
        best = df[df["status"].isin(["keep", "baseline"])]["val_bpb"].min()

        print(f"=== Experiment Progress ===")
        print(f"Total: {len(df)} | Kept: {len(kept)} | Discarded: {len(discarded)} | Crashed: {len(crashed)}")
        print(f"Best val_bpb: {best:.6f}" if best == best else "Best val_bpb: --")
        print()
        display(df[["exp", "description", "val_bpb", "tok_sec", "mfu", "status"]].tail(10))
    else:
        print("Waiting for results.tsv to be created...")

    # Check if agent is still running
    import subprocess
    result = subprocess.run(["pgrep", "-f", "tui.headless"], capture_output=True)
    if result.returncode != 0 and os.path.exists(RESULTS_PATH):
        print("\nAgent has finished.")
        break

    time.sleep(30)

In [ ]:
# Graceful stop -- sends SIGTERM so the current experiment finishes
import subprocess, signal, os

result = subprocess.run(["pgrep", "-f", "tui.headless"], capture_output=True, text=True)
if result.stdout.strip():
    pids = result.stdout.strip().split("\n")
    for pid in pids:
        os.kill(int(pid), signal.SIGTERM)
    print(f"Sent SIGTERM to agent PID(s): {', '.join(pids)}")
    print("Current experiment will finish, then agent will stop gracefully.")
else:
    print("No running agent process found.")

In [ ]:
# Final results summary
import pandas as pd

if os.path.exists(RESULTS_PATH):
    df = pd.read_csv(RESULTS_PATH, sep="\t")
    print(f"Total experiments: {len(df)}")
    print(f"Kept: {len(df[df['status'].isin(['keep', 'baseline'])])}")
    print(f"Discarded: {len(df[df['status'] == 'discard'])}")
    print(f"Crashed: {len(df[df['status'] == 'crash'])}")
    best_row = df.loc[df["val_bpb"].idxmin()]
    print(f"\nBest result:")
    print(f"  Experiment: {best_row['exp']}")
    print(f"  Description: {best_row['description']}")
    print(f"  val_bpb: {best_row['val_bpb']:.6f}")
    print(f"  tok/sec: {best_row['tok_sec']:,.0f}")
    print(f"  MFU: {best_row['mfu']:.1f}%")
    print(f"  GPU: {best_row['gpu_name']}")
else:
    print("No results file found. Run the experiment loop first.")